# Graph RAG — End to End Workshop

Build a Knowledge Graph from a research paper, load it into Neo4j, and query it using Graph RAG.

**Prerequisites:**
- Neo4j running: `docker compose -f workshop/docker-compose.yml up -d`
- Open Neo4j Browser: http://localhost:7474 (login: neo4j / workshop2024)

---

## Step 0: Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Load API key
load_dotenv(Path(".").resolve().parent / ".env")  # repo root
load_dotenv(Path(".").resolve() / ".env")  # workshop dir

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Neo4j connection
NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

# Helper function
def ask_llm(prompt, system="You are a helpful assistant."):
    return llm.invoke([SystemMessage(content=system), HumanMessage(content=prompt)]).content

print("Setup complete!")

: 

---
## Step 1: Read the Document

We're using a **real research paper from March 2026** — "Graphs RAG at Scale" (arXiv 2603.22340).  
This paper compares LPG, RDF, and Agentic RAG approaches. The LLM has **never seen this paper**.

In [ ]:
document = Path("../document-to-kg/data/ai_agents_paper.txt").read_text()

print(f"Document length: {len(document)} characters")
print(f"\nFirst 500 characters:\n")
print(document[:500])
print("\n...")

---
## Step 2: Define the Knowledge Graph Schema

We use **Pydantic models** to tell the LLM exactly what structure we want back.  
This is called **structured output** — the LLM returns data that matches our schema, not free text.

In [ ]:
class Entity(BaseModel):
    """A single entity (node) in the knowledge graph."""
    name: str = Field(description="Entity name")
    type: str = Field(description="PERSON, ORGANIZATION, TECHNOLOGY, MODEL, METHOD, DATABASE, or FRAMEWORK")
    description: str = Field(description="One-line description")

class Relationship(BaseModel):
    """A relationship (edge) between two entities."""
    source: str = Field(description="Source entity name")
    target: str = Field(description="Target entity name")
    type: str = Field(description="e.g. AUTHORED, USES, BUILT, STORED_IN")
    description: str = Field(description="One-line description")

class KnowledgeGraph(BaseModel):
    """Complete knowledge graph extracted from a document."""
    entities: list[Entity] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)

print("Schema defined!")
print(f"Entity fields: {list(Entity.model_fields.keys())}")
print(f"Relationship fields: {list(Relationship.model_fields.keys())}")

---
## Step 3: Extract Knowledge Graph from Document

We send the entire paper to the LLM and ask it to extract **entities** and **relationships**.  
Using `with_structured_output()` — the LLM returns a `KnowledgeGraph` Pydantic object, not free text.

In [ ]:
# Create a structured LLM that returns KnowledgeGraph objects
structured_llm = llm.with_structured_output(KnowledgeGraph)

print("Sending document to LLM for extraction... (takes ~10 seconds)")

kg = structured_llm.invoke([
    SystemMessage(content="""You are a knowledge graph extraction expert.
Extract entities (PERSON, ORGANIZATION, TECHNOLOGY, MODEL, METHOD, DATABASE, FRAMEWORK)
and relationships between them. Only extract what is explicitly stated."""),
    HumanMessage(content=f"Extract all entities and relationships:\n\n{document}"),
])

print(f"\nExtracted: {len(kg.entities)} entities, {len(kg.relationships)} relationships")

### View Extracted Entities

In [ ]:
print(f"{'TYPE':<15} {'NAME':<45} DESCRIPTION")
print("=" * 100)
for e in kg.entities:
    print(f"{e.type:<15} {e.name:<45} {e.description}")

### View Extracted Relationships

In [ ]:
print(f"{'SOURCE':<35} {'RELATION':<20} {'TARGET':<35}")
print("=" * 90)
for r in kg.relationships:
    print(f"{r.source:<35} {r.type:<20} {r.target:<35}")

---
## Step 4: Load into Neo4j

Now we take those entities and relationships and run **Cypher CREATE queries** to store them in Neo4j.  
This is the same Cypher you learned in Hour 1!

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

with driver.session() as s:
    # Clear old data
    s.run("MATCH (n) DETACH DELETE n")
    print("Cleared existing data\n")
    
    # Create nodes
    for e in kg.entities:
        s.run("CREATE (n:Entity {name: $name, type: $type, description: $desc})",
              name=e.name, type=e.type, desc=e.description)
        print(f"  Created node: [{e.type}] {e.name}")
    
    print()
    
    # Create relationships
    created = 0
    for r in kg.relationships:
        result = s.run(
            """MATCH (a:Entity {name: $src}), (b:Entity {name: $tgt})
               CREATE (a)-[:RELATES_TO {type: $rel, description: $desc}]->(b)
               RETURN count(*) AS c""",
            src=r.source, tgt=r.target, rel=r.type, desc=r.description)
        c = result.single()["c"]
        if c > 0:
            print(f"  Created edge: {r.source} --[{r.type}]--> {r.target}")
            created += 1
        else:
            print(f"  Skipped (node not found): {r.source} --[{r.type}]--> {r.target}")
    
    # Stats
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

print(f"\n{'='*50}")
print(f"Knowledge Graph loaded: {nodes} nodes, {edges} edges")
print(f"\nGo to http://localhost:7474 and run:")
print(f"  MATCH (n)-[r]->(m) RETURN n, r, m")

driver.close()

---
## Step 5: Visualize the Graph (inline)

Let's see the graph right here in the notebook using pyvis.

In [ ]:
from pyvis.network import Network
from IPython.display import IFrame
import os

COLORS = {
    "PERSON": "#FF6B6B", "ORGANIZATION": "#4ECDC4", "TECHNOLOGY": "#45B7D1",
    "MODEL": "#96CEB4", "METHOD": "#FFEAA7", "DATABASE": "#DDA0DD",
    "FRAMEWORK": "#F0E68C",
}

net = Network(height="600px", width="100%", directed=True, bgcolor="#ffffff",
              cdn_resources="remote")
net.barnes_hut(gravity=-3000)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    for r in s.run("MATCH (n:Entity) RETURN n.name AS name, n.type AS type, n.description AS desc"):
        color = COLORS.get(r["type"], "#DFE6E9")
        net.add_node(r["name"], label=r["name"], color=color, title=f"{r['type']}: {r['desc']}", size=20)
    for r in s.run("MATCH (a)-[r]->(b) RETURN a.name AS src, b.name AS tgt, r.type AS type"):
        net.add_edge(r["src"], r["tgt"], label=r["type"], color="#B2BEC3")
driver.close()

html_path = os.path.abspath("kg_visualization.html")
net.save_graph(html_path)
print(f"Graph saved to: {html_path}")
print("If it doesn't render below, open the HTML file in your browser.")
IFrame(src="kg_visualization.html", width="100%", height=620)

---
## Step 6: Graph RAG — Ask Questions!

This is the payoff. The Graph RAG pipeline:

1. **LLM Call 1**: Convert user question → Cypher query (using graph schema)
2. **Run Cypher** on Neo4j → get relevant subgraph
3. **LLM Call 2**: Answer the question using graph results as context

In [ ]:
def graph_rag(question):
    """Complete Graph RAG pipeline: Question → Cypher → Neo4j → Answer"""
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    
    # Get entity names so LLM knows what's in the graph
    with driver.session() as s:
        names = [r["name"] for r in s.run("MATCH (n:Entity) RETURN n.name AS name")]
    
    # LLM CALL 1: Question → Cypher
    cypher = ask_llm(
        prompt=f"Convert to Cypher:\n\n{question}",
        system=f"""Neo4j Cypher expert. Schema:
- Nodes: :Entity (name, type, description)
- Relationships: :RELATES_TO (type, description)
Entities in graph: {names}
Use toLower() and CONTAINS for matching. Return ONLY Cypher, no markdown."""
    ).strip().replace("```cypher", "").replace("```", "").strip()
    
    print(f"Generated Cypher: {cypher}\n")
    
    # Run Cypher
    try:
        with driver.session() as s:
            rows = [dict(r) for r in s.run(cypher)]
    except:
        rows = []
    
    if not rows:
        with driver.session() as s:
            rows = [dict(r) for r in s.run(
                "MATCH (a:Entity)-[r]->(b:Entity) RETURN a.name AS source, r.type AS rel, b.name AS target"
            )]
    
    driver.close()
    print(f"Graph results ({len(rows)} rows):")
    for row in rows:
        print(f"  {row}")
    
    # LLM CALL 2: Graph results → Answer
    context = "\n".join([str(r) for r in rows])
    answer = ask_llm(
        prompt=f"Graph results:\n{context}\n\nQuestion: {question}",
        system="Answer using ONLY the graph results. Cite the relationships."
    )
    return answer

print("graph_rag() function ready!")

### Query 1: Simple fact

In [ ]:
answer = graph_rag("Who are the authors and what organization are they from?")
print(f"\nAnswer: {answer}")

### Query 2: Relationship question

In [ ]:
answer = graph_rag("What is the relationship between BGE-m3 and the RAG2 pipeline?")
print(f"\nAnswer: {answer}")

### Query 3: Multi-hop

In [ ]:
answer = graph_rag("What technologies does the RDF pipeline use?")
print(f"\nAnswer: {answer}")

### Query 4: Comparison

In [ ]:
answer = graph_rag("Compare LPG and RDF — which performed better?")
print(f"\nAnswer: {answer}")

---
## Summary

What we did:

```
Document (research paper)
    │
    ▼
LLM extracts entities + relationships  (structured output)
    │
    ▼
Cypher CREATE queries load into Neo4j   (same syntax from Hour 1)
    │
    ▼
User asks a question
    │
    ├── LLM Call 1: Question → Cypher query
    ├── Run Cypher on Neo4j → get subgraph
    └── LLM Call 2: Subgraph + Question → Answer
```

**Why this beats classic RAG:**
- Classic RAG finds similar text chunks → loses relationships between entities
- Graph RAG stores relationships explicitly → can traverse connections to answer complex questions